# 深度學習框架應用（大學部）｜主題 5：卷積神經網路（CNN）原理

這本筆記本配合投影片《主題 5：卷積神經網路（CNN）原理》：先用公式算尺寸和參數量，再親手做一個卷積核、比較三種池化，接著搭一個小 CNN，最後拿 Fashion-MNIST（10 類衣服鞋子的灰階照片）真的訓練一次，看第一層卷積核學到什麼。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 輸出尺寸與參數量（先手算、再驗證） | 第 90 頁 | 第 30、31、57 頁 |
| 二 | 手刻 3×3 卷積核，找出直的邊 | 第 91 頁 | 第 21、22、23 頁 |
| 三 | 三種池化比一比 | 第 92 頁 | 第 37、39、41 頁 |
| 四 | 搭一個小卷積網路，印出每一層的形狀 | 第 93 頁 | 第 34、55、58 頁 |
| 五 | 在 Fashion-MNIST 上訓練，畫出第一層卷積核 | 第 94 頁 | 第 74、77、78 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W05_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 練習一～四都是小張量，CPU 就跑得動。練習五要訓練：有 GPU 就開（「執行階段 › 變更執行階段類型 › T4 GPU」），沒有也沒關係，CPU 大約幾分鐘跑完。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

五題都會用到下面這些套件，先跑這一格。`count(m)` 是一個小工具：數一層（或整個模型）有幾個參數，練習一、四會用到。

In [ ]:
# ===== 環境準備 =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
dev = 'cuda' if torch.cuda.is_available() else 'cpu'


def count(m):
    """數一個模組（一層或整個模型）有幾個參數。"""
    return sum(p.numel() for p in m.parameters())


print('PyTorch：', torch.__version__, '｜裝置：', dev)

## 練習一：輸出尺寸與參數量 (20%)

> 題目在投影片**第 90 頁**｜觀念在第 30、31、57 頁

**情境**：設計 CNN 時，最常當機的原因是「形狀對不上」。主題 5 的做法是：先用公式手算每一層的輸出邊長和參數量，再讓程式印出來對答案。

**這題在練**：用輸出尺寸公式 O = ⌊(I + 2P − K) / S⌋ + 1 和參數量公式 K × K × C_in × C_out + C_out，算出兩層卷積的形狀和參數，再用 `nn.Conv2d` 驗證

**步驟**

1. 把公式寫成兩個小函式：`out_size()` 填 P 和 S 的位置，`n_params()` 填輸入通道和偏置的個數
2. 手算三個尺寸：28×28 用 3×3、P = 1；28×28 用 2×2、S = 2（池化）；14×14 用 3×3、S = 2
3. 建兩層真的 `nn.Conv2d`，讓一張 28×28 的假照片走過去，對照手算和程式印出的形狀、參數量

> **小提醒**：`//` 是「除完無條件捨去」，正好就是公式裡的 ⌊ ⌋。

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：先手算尺寸和參數量，再讓程式對答案
def out_size(I, K, P=0, S=1):         # 輸出邊長
    return (I + 2 * ___ - K) // ___ + 1  # ✏️ 尺寸公式
def n_params(K, cin, cout):           # 一層卷積的參數量
    return K * K * ___ * cout + ___   # ✏️ 權重＋偏置
print('手算尺寸：', out_size(28, 3, P=1),
      out_size(28, 2, S=2), out_size(14, 3, S=2))
print('手算參數：', n_params(3, 1, 8), n_params(3, 8, 16))
c1 = nn.Conv2d(1, 8, 3, padding=1)    # 28 → ?
c2 = nn.Conv2d(8, 16, 3, stride=2)    # 池化後 14 → ?
h1 = c1(torch.zeros(1, 1, 28, 28))    # 一張假照片
h2 = c2(nn.MaxPool2d(2)(h1))
print('程式形狀：', tuple(h1.shape), tuple(h2.shape))
print('程式參數：', count(c1), count(c2))

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：四行結果（手算尺寸、手算參數、程式形狀、程式參數）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. out_size(14, 3, S=2)：(14 − 3) ÷ 2 = 5.5，為什麼答案是 6？用「數位置」的方法說明。**

（在這裡寫）

**2. c2 的 1,168 個參數怎麼來？為什麼跟輸入的照片是 28×28 還是 280×280 都沒有關係？**

（在這裡寫）

## 練習二：手刻 3×3 卷積核，找出直的邊 (20%)

> 題目在投影片**第 91 頁**｜觀念在第 21、22、23 頁

**情境**：一張 5×5 的灰階小圖，左邊三行是 0（暗），右邊兩行是 9（亮），中間有一條直直的明暗交界。

**這題在練**：自己設計卷積核的數字，親眼看到「直邊核」在交界處亮起來、「橫邊核」完全沒反應，並用 padding 讓輸出維持 5×5

**步驟**

1. 填好直邊核 kv：每一列都是「左邊 −1、中間 0、右邊 +1」，意思是右邊減左邊
2. 用 `.T`（轉置）把直邊核轉成橫邊核 kh，兩個核各掃一次，印出特徵圖
3. 補一圈零（padding）再掃一次，讓輸出的邊長和輸入一樣是 5

> `F.conv2d(輸入, 核)` 是不帶參數的卷積函式；核的形狀要是 (輸出通道, 輸入通道, 高, 寬)，所以用 `.view(1, 1, 3, 3)` 把 3×3 的核包成四維。

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習二：手刻 3×3 卷積核，找出直的邊
img = torch.zeros(1, 1, 5, 5)  # (張數, 通道, 高, 寬)
img[0, 0, :, 3:] = 9.          # 右邊兩行變亮
kv = torch.tensor([[___, 0., ___]] * 3)  # ✏️ 右減左
kh = kv.___                    # ✏️ 轉成橫的
for name, k in [('直邊核', kv), ('橫邊核', kh)]:
    out = F.conv2d(img, k.view(1, 1, 3, 3))  # 不補零
    print(name, tuple(out.shape))
    print(out[0, 0])
o = F.conv2d(img, kv.view(1, 1, 3, 3), padding=___)  # ✏️
print('補一圈零之後：', tuple(o.shape))

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：兩個核各自的特徵圖（形狀和數字）、補零之後的形狀

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 手算直邊核特徵圖第一列的三個數字（寫出其中一格的算式），和程式印出的結果對答案。**

（在這裡寫）

**2. 橫邊核為什麼整張特徵圖都是 0？這說明卷積核在做什麼？**

（在這裡寫）

## 練習三：三種池化比一比 (20%)

> 題目在投影片**第 92 頁**｜觀念在第 37、39、41 頁

**情境**：一張 4×4 的特徵圖，切成四塊 2×2：左上 [2, 0, 7, 3]、右上 [1, 4, 0, 1]、左下 [1, 1, 0, 2]、右下 [5, 2, 3, 6]。

**這題在練**：用最大池化、平均池化、全域平均池化各算一次，看懂三種池化留下了什麼、丟掉了什麼，以及池化對小幅位移的反應

**步驟**

1. 用 `nn.MaxPool2d(2)`、`nn.AvgPool2d(2)` 各算一次，再用 `nn.AdaptiveAvgPool2d(1)` 把整張壓成一個數
2. 把第 1、2 列對調（每個數都還在原來那一塊裡），看最大池化的結果有沒有變
3. 最後讓一張 7×7 的圖過最大池化，用尺寸公式先猜輸出邊長，再看程式印出什麼

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習三：三種池化比一比
fm = torch.tensor([[2., 0., 1., 4.],
                   [7., 3., 0., 1.],
                   [1., 1., 5., 2.],
                   [0., 2., 3., 6.]]).view(1, 1, 4, 4)
mx = nn.___(2)                    # ✏️ 2×2 取最大
av = nn.___(2)                    # ✏️ 2×2 取平均
gap = nn.AdaptiveAvgPool2d(___)   # ✏️ 整張剩幾格
print('Max：', mx(fm)[0, 0].tolist())
print('Avg：', av(fm)[0, 0].tolist())
print('GAP：', gap(fm).item())
moved = fm[..., [1, 0, 2, 3], :]  # 第 1、2 列對調
print('對調後的 Max：', mx(moved)[0, 0].tolist())
print('7×7 做 Max：', mx(torch.zeros(1, 1, 7, 7)).shape)

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：六行結果（Max、Avg、GAP、對調後的 Max、7×7 的輸出形狀）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 左上那塊 [2, 0, 7, 3]，最大池化留下 7、平均池化是 3。哪一個比較能保留「這塊裡有一個很強的反應」？為什麼對調兩列之後，最大池化的結果完全沒變？**

（在這裡寫）

**2. GAP 把整張 4×4 壓成 2.375 一個數，丟掉了什麼？7×7 的圖做 MaxPool2d(2) 為什麼是 3×3？**

（在這裡寫）

## 練習四：搭一個小卷積網路，印出每一層的形狀 (20%)

> 題目在投影片**第 93 頁**｜觀念在第 34、55、58 頁

**情境**：Fashion-MNIST 是 28×28 的灰階照片（1 個通道），分成 10 類衣服和鞋子。我們要搭一個「卷積 → ReLU → 池化」兩次，再接一層卷積、攤平、分類頭的小卷積網路（CNN）。

**這題在練**：自己決定 padding、輸入通道和攤平後的大小，讓模型的形狀一路接得起來，再比較「攤平」和「全域平均池化」兩種分類頭的參數量

**步驟**

1. 先執行「準備」那一格，它會定義 `show_layers()`：一層一層印出輸出形狀和參數量
2. 填好 ✏️ 格的空格：padding 要讓邊長不變、第二個 block 的輸入通道、攤平之後有幾個數
3. 看印出來的表，對照手算；再算 GAP 版的分類頭只要幾個參數

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改）=====
def show_layers(model, x):
    """把 x 一層一層傳下去，印出每一層的輸出形狀和這一層的參數量。"""
    print(f'{"層":<16}{"輸出形狀":<16}參數量')
    for layer in model:
        x = layer(x)
        print(f'{type(layer).__name__:<18}{str(tuple(x.shape)):<20}{count(layer):>7,}')
    print(f'總參數量：{count(model):,}')


print('show_layers() 準備好了')

In [ ]:
# ✏️ 練習四：搭一個小卷積網路，印出每一層的形狀
def block(i, o, k):                 # 卷積 → ReLU → 池化
    return [nn.Conv2d(i, o, k, padding=___),  # ✏️
            nn.ReLU(), nn.MaxPool2d(2)]
model = nn.Sequential(
    *block(1, 8, 5), *block(___, 16, 3),  # ✏️ 輸入通道
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
    nn.Flatten(),
    nn.Linear(32 * ___ * ___, 10))  # ✏️ 攤平後幾個數
show_layers(model, torch.zeros(2, 1, 28, 28))  # 兩張假圖
gap = nn.AdaptiveAvgPool2d(___)     # ✏️ 全域平均
print('GAP 之後：', gap(torch.zeros(2, 32, 7, 7)).shape)
print('攤平版分類頭：', count(model[-1]))
print('GAP 版分類頭：', count(nn.Linear(32, 10)))

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：每一層的輸出形狀與參數量、總參數量、GAP 之後的形狀、兩種分類頭的參數量

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 第二個 block 的輸入通道為什麼是 8？padding 為什麼寫成 k // 2？**

（在這裡寫）

**2. 攤平版分類頭 15,690 個參數，GAP 版只要 330 個，差幾倍？整個模型的參數大部分在哪裡？**

（在這裡寫）

## 練習五：在 Fashion-MNIST 上訓練，畫出第一層卷積核 (20%)

> 題目在投影片**第 94 頁**｜觀念在第 74、77、78 頁

**情境**：把練習四搭好的小 CNN 拿去 Fashion-MNIST 真的訓練：訓練集 60,000 張、測試集 10,000 張，練 3 個 epoch。

**這題在練**：把主題 3、4 學過的訓練迴圈套到 CNN 上，看每一類各答對幾成，再把第一層的 8 個 5×5 卷積核畫出來，看它自己學到什麼

**步驟**

1. 先執行「準備資料」那一格：下載資料、定義 `evaluate()`、`per_class()`、`show_kernels()`，並把練習四的模型重設成亂猜的起點
2. 填好 ✏️ 格的空格：損失函數、切到訓練模式、反向傳播、第一層卷積在 Sequential 裡的編號
3. 看三個 epoch 的測試準確率、最低的是哪一類，以及 8 個卷積核長什麼樣子

> 想重跑 ✏️ 格，要先重跑「準備資料」，模型才會回到亂猜的起點。CPU 上 3 個 epoch 大約一兩分鐘到幾分鐘，有 GPU 會更快。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備資料（直接執行，不用改）=====
# 第一次執行會下載 Fashion-MNIST（約 30 MB），等十幾秒就好
tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])  # 0～255 → 0～1
train_set = datasets.FashionMNIST('./data', train=True, transform=tf, download=True)
test_set = datasets.FashionMNIST('./data', train=False, transform=tf, download=True)
train_dl = DataLoader(train_set, batch_size=128, shuffle=True)
test_dl = DataLoader(test_set, batch_size=512)
print(f'訓練 {len(train_set)} 張、測試 {len(test_set)} 張，一張的形狀 {tuple(train_set[0][0].shape)}')
print('10 類：', train_set.classes)


def evaluate(model, dl):
    """切到 eval 模式，算整個資料集的準確率。"""
    model.eval(); hit = 0
    with torch.no_grad():
        for x, y in dl:
            hit += (model(x.to(dev)).argmax(1).cpu() == y).sum().item()
    return hit / len(dl.dataset)


def per_class(model, dl):
    """每一類各答對幾成，從最低排到最高。"""
    model.eval(); hit, tot = torch.zeros(10), torch.zeros(10)
    with torch.no_grad():
        for x, y in dl:
            p = model(x.to(dev)).argmax(1).cpu()
            for c in range(10):
                m = (y == c)                      # 只留下第 c 類的照片
                hit[c] += (p[m] == c).sum(); tot[c] += m.sum()
    acc = hit / tot
    for c in acc.argsort().tolist():
        print(f'{dl.dataset.classes[c]:<12} {acc[c]:.3f}')


def show_kernels(w):
    """把第一層的卷積核排成一排小圖：先縮到 0～1，亮＝正、暗＝負。"""
    w = w.detach().cpu()
    print('第一層卷積核的形狀：', tuple(w.shape))
    w = (w - w.min()) / (w.max() - w.min())
    fig, axs = plt.subplots(1, w.shape[0], figsize=(w.shape[0] * 1.2, 1.6))
    for k, ax in enumerate(axs):
        ax.imshow(w[k, 0], cmap='gray'); ax.set_title(f'#{k}'); ax.axis('off')
    plt.show()


if 'model' in globals():                     # 用練習四搭好的模型
    torch.manual_seed(0)                     # 固定亂數，大家從同一個起點開始練
    for m in model.modules():
        if hasattr(m, 'reset_parameters'):
            m.reset_parameters()             # 把權重重新設成亂猜的起點
    model = model.to(dev)
    print('模型已重設，參數量：', count(model))
else:
    print('還沒有 model：請先完成練習四，再回來執行這一格')

In [ ]:
# ✏️ 練習五：在 Fashion-MNIST 上訓練 3 個 epoch
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.___()                  # ✏️ 多類別分類
for ep in range(3):
    model.___()                     # ✏️ 切到訓練模式
    for x, y in train_dl:
        loss = loss_fn(model(x.to(dev)), y.to(dev))
        opt.zero_grad()
        loss.___()                  # ✏️ 反向傳播
        opt.step()
    acc = evaluate(model, test_dl)  # 會切到 eval 模式
    print(f'epoch {ep + 1}：測試準確率 {acc:.3f}')
per_class(model, test_dl)           # 每一類各答對幾成
show_kernels(model[___].weight)     # ✏️ 第一層卷積

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行測試準確率、每一類的準確率、8 個卷積核的圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 你的三個 epoch 測試準確率各是多少？哪一類最低、它最容易被認成哪幾類？**

（在這裡寫）

**2. 8 個卷積核的圖大概長什麼樣子？為什麼說第一層學到的是「通用的」特徵？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：padding 為什麼要乘 2） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W05_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。